In [10]:
import torch
import torchvision.transforms as transforms
import torchvision.models as models
from PIL import Image
import matplotlib.pyplot as plt
import numpy as np
import operator, math, os, glob
import torch.nn as nn
from matplotlib.pyplot import imread

In [11]:
output_folder = "VGG16" # @param ["VIT","ResNet50","EfficientNet","VGG16"]
model_name="vgg16" # @param ["vit_b_16","resnet50","efficientnet","vgg16"]

In [12]:
def euclidean_distance(vec1, vec2):
    return np.sqrt(np.sum((vec1 - vec2) ** 2))

In [ ]:
def vitmodel(img_path):
        vit = models.vit_b_16(weights=models.ViT_B_16_Weights.DEFAULT)
        device = torch.device('mps' if torch.backends.mps.is_available() else ('cuda' if torch.cuda.is_available() else 'cpu'))

        img = Image.open(img_path).convert("RGB")
        vit=vit.to(device)

        # utilisation de la transformation par défaut pour le modèle VIT
        preprocessing = models.ViT_B_16_Weights.DEFAULT.transforms()

        img = preprocessing(img).to(device)

        # ajout d'une dimension batch
        img = img.unsqueeze(0)
        vit.eval()
        with torch.no_grad():
                features = vit._process_input(img)

                # extension du token CLS à la dimension batch
                batch_class_token = vit.class_token.expand(img.shape[0], -1, -1)
                features = torch.cat([batch_class_token, features], dim=1)

                features = vit.encoder(features)

                # nous sommes intéressés par la représentation du token CLS que nous avons ajouté à la position 0
                features = features[:, 0]
                return features.cpu().detach().numpy().squeeze().flatten()


In [14]:
transform = transforms.Compose([
    transforms.Resize((224, 224)),  # Taille standard de VGG16
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

In [ ]:
def extract_features(img_path, model_name):
    img = Image.open(img_path).convert("RGB")     # Charger l'image
    #device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    device = torch.device('mps' if torch.backends.mps.is_available() else ('cuda' if torch.cuda.is_available() else 'cpu'))

    image = transform(img).unsqueeze(0).to(device)  # Appliquer les transformations et ajouter une dimension batch
    
    return vitmodel(img_path)

    feature_extractor.eval()  # Mode évaluation
    feature_extractor=feature_extractor.to(device)
    with torch.no_grad():                             # Désactiver le calcul des gradients
        features = feature_extractor(image)
    return features.cpu().numpy().squeeze().flatten()


In [16]:
input_folder = "MIR_DATASETS_B" 
os.makedirs(output_folder, exist_ok=True)  


In [17]:

for filename in os.listdir(input_folder):
    if filename.lower().endswith((".png", ".jpg", ".jpeg")):  # Vérifier le format de l'image
        image_path = os.path.join(input_folder, filename)
        features = extract_features(image_path,model_name)

        # Enregistrer les features sous forme de fichier texte
        output_path = os.path.join(output_folder, filename.split('.')[0] + ".txt")
        np.savetxt(output_path, features, fmt="%.6f")  # Format en 6 décimales

        print(f"Features extraites et enregistrées : {output_path}")

print("Extraction terminée !")

Features extraites et enregistrées : VGG16/3_2_poissons_guitarfish_2990.txt
Features extraites et enregistrées : VGG16/1_3_chiens_Chihuahua_1378.txt
Features extraites et enregistrées : VGG16/0_1_araignees_wolfspider_183.txt
Features extraites et enregistrées : VGG16/2_2_oiseaux_greatgreyowl_2193.txt
Features extraites et enregistrées : VGG16/4_4_singes_orangutan_4284.txt
Features extraites et enregistrées : VGG16/1_0_chiens_Siberianhusky_828.txt
Features extraites et enregistrées : VGG16/2_5_oiseaux_bulbul_2621.txt
Features extraites et enregistrées : VGG16/2_5_oiseaux_bulbul_2635.txt
Features extraites et enregistrées : VGG16/2_2_oiseaux_greatgreyowl_2187.txt
Features extraites et enregistrées : VGG16/4_4_singes_orangutan_4290.txt
Features extraites et enregistrées : VGG16/3_2_poissons_guitarfish_2984.txt
Features extraites et enregistrées : VGG16/0_1_araignees_wolfspider_197.txt
Features extraites et enregistrées : VGG16/4_1_singes_chimpanzee_3880.txt
Features extraites et enregistr

KeyboardInterrupt: 